# Análisis exploratorio de los tiempos logísticos del transporte de carga en Colombia

**Esperas y operaciones de cargue y descargue** · Estadística I · Ingeniería Industrial · UIS

Autores: Luis Felipe Acuña, Miguel Velandia y David Mateus.

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/david2240092-maker/eda-tiempos-logisticos-rndc/blob/main/EDA_Tiempos_Logisticos_Colab.ipynb)

Datos: *Tiempos logísticos de cada viaje de vehículos de carga* (RNDC, Ministerio de Transporte), publicado en
[datos.gov.co](https://www.datos.gov.co/d/tfrd-amb4). Cada fila es una remesa de un viaje de enero de 2021.

**Cómo usarlo:** menú *Entorno de ejecución → Ejecutar todo*. La celda 2 intenta descargar el CSV desde
datos.gov.co; si no puede, te pedirá que lo subas desde tu computador (pesa unos 320 MB).

Contenido: 1) carga, 2) limpieza, 3) atípicos, 4) estadística descriptiva, 5) histogramas, 6) carga y vehículo,
7) municipios, 8) día y hora, 9) correlaciones, 10) empresas y valor, 11) sensibilidad, 12) conclusiones.

In [ ]:
# 1. Librerías y estilo de las gráficas
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch
from scipy import stats

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
INK, INK2, GRID, GRAY = "#0b0b0b", "#52514e", "#e3e2de", "#a9a8a3"
plt.rcParams.update({
    "font.family": "serif", "font.size": 10, "axes.titlesize": 10.5, "axes.labelsize": 10,
    "xtick.labelsize": 9, "ytick.labelsize": 9, "legend.fontsize": 9,
    "axes.edgecolor": INK2, "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": INK2, "ytick.color": INK2,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.axisbelow": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "figure.dpi": 110,
})
TIMES = ["HORAS_ESPERA_CARGUE", "HORAS_CARGUE", "HORAS_ESPERA_DESCARGUE", "HORAS_DESCARGUE"]
LAB = {"HORAS_ESPERA_CARGUE": "Espera para cargue", "HORAS_CARGUE": "Cargue",
       "HORAS_ESPERA_DESCARGUE": "Espera para descargue", "HORAS_DESCARGUE": "Descargue",
       "HORAS_VIAJE": "Horas de viaje", "espera_total": "Espera total (cargue + descargue)"}
DIAS = ["Lun", "Mar", "Mié", "Jue", "Vie", "Sáb", "Dom"]


def pf(p):
    # valor p legible: los muy pequeños se muestran como < 0.001
    return "< 0.001" if p < 1e-3 else f"= {p:.3f}"


def kruskal(var, col, d, grupos=None):
    """Kruskal-Wallis con tamaño de efecto epsilon^2 = H / (n - 1)."""
    d = d.dropna(subset=[col])
    if grupos is not None:
        d = d[d[col].isin(grupos)]
    gs = [g[var].values for _, g in d.groupby(col)]
    H, p = stats.kruskal(*gs)
    return {"H": H, "p": p, "eps2": H / (len(d) - 1), "n": len(d), "grupos": len(gs)}

## 1. Carga de los datos

Se lee todo como texto y luego se convierten las columnas numéricas y las fechas. Las placas y los conductores
**no se muestran ni se analizan**: solo se usan para detectar registros repetidos.

In [ ]:
# 2. Obtener el CSV (descarga directa; si falla, subirlo a mano)
CSV = "tiempos_logisticos.csv"
URL = "https://www.datos.gov.co/api/views/tfrd-amb4/rows.csv?accessType=DOWNLOAD"

if not os.path.exists(CSV):
    try:
        import shutil, urllib.request
        print("Descargando desde datos.gov.co (puede tardar unos minutos)...")
        req = urllib.request.Request(URL, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req) as r, open(CSV, "wb") as f:
            shutil.copyfileobj(r, f)
        with open(CSV, encoding="utf-8") as f:       # comprobar que sea el CSV y no una página de error
            assert "HORAS_ESPERA_CARGUE" in f.readline()
    except Exception as e:
        print("No se pudo descargar:", e)
        print("Sube el archivo CSV desde tu computador:")
        from google.colab import files
        subido = files.upload()
        CSV = list(subido.keys())[0]
print("Archivo:", CSV, f"({os.path.getsize(CSV) / 1e6:,.0f} MB)")

In [ ]:
# 3. Lectura y conversión de tipos
raw = pd.read_csv(CSV, dtype=str, low_memory=False)
N0 = len(raw)

NUM = ["HORAS_VIAJE", "HORAS_ESPERA_CARGUE", "HORAS_CARGUE", "HORAS_ESPERA_DESCARGUE",
       "HORAS_DESCARGUE", "VALOR_PACTADO", "VALOR_PAGADO", "CANTIDAD", "CANTIDAD_REMESAS_VIAJE"]
for c in NUM:
    raw[c] = pd.to_numeric(raw[c].str.replace(",", ""), errors="coerce")
def leer_fecha(s):
    # Formato del portal ("2021 Jan 26 12:00:00 AM"); si el archivo trae otro (ISO, dd/mm/aaaa), se infiere.
    out = pd.to_datetime(s, format="%Y %b %d %I:%M:%S %p", errors="coerce")
    if out.isna().mean() > 0.5:
        out = pd.to_datetime(s, errors="coerce", format="mixed")
    return out

raw["fs"] = leer_fecha(raw.FECHASALIDACARGUE)
raw["fl"] = leer_fecha(raw.FECHALLEGADADESCARGUE)
assert raw.fs.notna().mean() > 0.9, f"No se pudieron leer las fechas. Ejemplo: {raw.FECHASALIDACARGUE.iloc[0]!r}"

claves_viaje = ["PLACA", "FECHASALIDACARGUE", "CODIGO_CARGUE", "CODIGO_DESCARGUE"]
print(f"Registros (remesas):        {N0:,}")
print(f"Viajes aproximados:         {raw.groupby(claves_viaje).ngroups:,}")
print(f"Vehículos distintos:        {raw.PLACA.nunique():,}")
print(f"Empresas de transporte:     {raw.EMPRESA_TRANSPORTE.nunique():,}")
print(f"Salidas entre:              {raw.fs.min().date()} y {raw.fs.max().date()}")
print(f"Llegadas hasta:             {raw.fl.max().date()}")
raw[["NATURALEZA", "CARGUE", "DESCARGUE", "CONFIGURACION"] + TIMES].head(5)

## 2. Limpieza de datos

Cuatro reglas, en este orden:

1. quitar registros **duplicados** exactos;
2. quitar registros con algún campo **vacío** (naturaleza, unidad, destino, configuración o tiempos);
3. quitar registros con **horas de cargue negativas** (inválidas);
4. quitar **valores imposibles**: algún tiempo de espera u operación de más de 168 h (una semana) o más de 720 h de viaje (30 días).

Los **atípicos por la regla del IQR no se eliminan**: en variables cuya mediana es casi cero, esa regla marca
como atípica cualquier espera de más de unas 2 h, y esas esperas son justamente lo que se estudia.

In [ ]:
# 4. Limpieza
def limpiar(d, dedup=True):
    log = {}
    if dedup:
        n = len(d); d = d.drop_duplicates(); log["Duplicados exactos"] = n - len(d)
    n = len(d)
    d = d.dropna(subset=["NATURALEZA", "UNID_MEDIDA", "DESCARGUE", "CONFIGURACION"] + NUM[:5])
    log["Campos vacíos"] = n - len(d)
    n = len(d); d = d[d.HORAS_CARGUE >= 0]; log["Cargue negativo"] = n - len(d)
    n = len(d)
    d = d[~((d[TIMES] > 168).any(axis=1) | (d.HORAS_VIAJE > 720))]
    log["Tiempos imposibles"] = n - len(d)
    return d.copy(), log


df, LOG = limpiar(raw)
dfB, _ = limpiar(raw, dedup=False)     # variante para el análisis de sensibilidad (con duplicados)

resumen_limpieza = pd.Series(LOG, name="Registros eliminados").to_frame()
resumen_limpieza.loc["Total eliminado"] = resumen_limpieza.sum()
display(resumen_limpieza.style.format("{:,.0f}"))
print(f"Quedan {len(df):,} de {N0:,} registros ({(1 - len(df) / N0) * 100:.1f} % menos).")

In [ ]:
# 5. Variables derivadas
df["espera_total"] = df.HORAS_ESPERA_CARGUE + df.HORAS_ESPERA_DESCARGUE
df["t_nodo"] = df[TIMES].sum(axis=1)
df["frac_espera"] = np.where(df.t_nodo > 0, df.espera_total / df.t_nodo, np.nan)
df["ec2"] = df.HORAS_ESPERA_CARGUE > 2        # espera prolongada para cargue (> 2 h)
df["ed2"] = df.HORAS_ESPERA_DESCARGUE > 2     # espera prolongada para descargue (> 2 h)
df["dow_s"] = df.fs.dt.dayofweek               # día de la semana de la salida (0 = lunes)
df["dow_l"] = df.fl.dt.dayofweek               # día de la semana de la llegada
df["h_ll"] = df.HORA_LLEGADA_DESCARGUE.str[11:13].astype(int)   # hora de llegada al descargue
print("Espera > 2 h para cargue:   ", f"{df.ec2.mean() * 100:.1f} %", f"({df.ec2.sum():,} registros)")
print("Espera > 2 h para descargue:", f"{df.ed2.mean() * 100:.1f} %", f"({df.ed2.sum():,} registros)")

## 3. Atípicos (regla del IQR)

Se calcula la cerca superior `Q3 + 1,5·IQR` de cada variable de tiempo y se cuenta cuántos registros la superan.
Solo se **miden**, no se eliminan.

In [ ]:
# 6. Atípicos por IQR
fences, filas = {}, []
for c in TIMES:
    q1, q3 = df[c].quantile([.25, .75])
    f = q3 + 1.5 * (q3 - q1)
    fences[c] = f
    filas.append([LAB[c], f, int((df[c] > f).sum()), (df[c] > f).mean() * 100])
tabla_iqr = pd.DataFrame(filas, columns=["Variable", "Cerca superior (h)", "Atípicos", "% de registros"]).set_index("Variable")
display(tabla_iqr.style.format({"Cerca superior (h)": "{:.2f}", "Atípicos": "{:,.0f}", "% de registros": "{:.1f}"}))
out_any = (df[TIMES] > pd.Series(fences)).any(axis=1)
print(f"Registros atípicos en al menos una variable: {out_any.mean() * 100:.1f} %")

## 4. Estadística descriptiva

Media, mediana, moda, varianza, percentiles y asimetría de las variables de tiempo, y la proporción de registros con
espera cero, mayor a 2 h y mayor a 12 h.

In [ ]:
# 7. Tabla descriptiva
filas = []
for c in TIMES + ["HORAS_VIAJE", "espera_total"]:
    s = df[c]
    filas.append([LAB[c], s.mean(), s.median(), s.mode().iloc[0], s.var(), s.quantile(.25), s.quantile(.75),
                  s.quantile(.95), s.quantile(.99), s.skew()])
desc = pd.DataFrame(filas, columns=["Variable (h)", "Media", "Mediana", "Moda", "Varianza", "P25", "P75", "P95", "P99",
                                    "Asimetría"]).set_index("Variable (h)")
display(desc)

filas = []
for c in TIMES:
    s = df[c]
    filas.append([LAB[c], (s == 0).mean() * 100, (s > 2).mean() * 100, (s > 12).mean() * 100,
                  ((s % 1) == 0).mean() * 100])
display(pd.DataFrame(filas, columns=["Variable", "% = 0 h", "% > 2 h", "% > 12 h", "% horas enteras exactas"]).set_index("Variable"))

print(f"Sin espera en ningún sitio:          {((df.HORAS_ESPERA_CARGUE == 0) & (df.HORAS_ESPERA_DESCARGUE == 0)).mean() * 100:.1f} %")
print(f"Espera > 2 h en al menos un sitio:   {(df.ec2 | df.ed2).mean() * 100:.1f} %")
print(f"Espera > 2 h en ambos sitios:        {(df.ec2 & df.ed2).mean() * 100:.1f} %")
fe = df.frac_espera.dropna()
print(f"Fracción de espera dentro del tiempo en nodos: media {fe.mean() * 100:.1f} %, mediana {fe.median() * 100:.1f} %")

**Lectura:** en el registro típico no hay espera (mediana cercana a 0 h), pero las distribuciones tienen una cola
derecha muy marcada (asimetría alta): la media supera la mediana y el percentil 99 llega a varias horas.
Además, más de la mitad de las esperas son horas enteras exactas, lo que indica **redondeo en el reporte**.

## 5. Histogramas

Se grafican los valores hasta 12 h; la línea discontinua es la mediana y el recuadro indica qué porcentaje queda por encima de 12 h.

In [ ]:
# 8. Histogramas de las cuatro variables de tiempo
fig, axs = plt.subplots(2, 2, figsize=(8, 6.2))
bins = np.arange(0, 12.5, 0.5)
for ax, c in zip(axs.ravel(), TIMES):
    s = df[c]
    w = np.full(len(s[s <= 12]), 100 / len(s))
    ax.hist(s[s <= 12], bins=bins, weights=w, color=BLUE, edgecolor="white", linewidth=0.4)
    ax.axvline(s.median(), color=INK2, ls="--", lw=1)
    ax.set_title(LAB[c]); ax.set_xlim(0, 12)
    ax.text(0.97, 0.93, f"> 12 h: {(s > 12).mean() * 100:.1f} %", transform=ax.transAxes, ha="right", va="top", fontsize=8.5, color=INK2)
    ax.text(0.97, 0.80, f"mediana {s.median():.2f} h", transform=ax.transAxes, ha="right", va="top", fontsize=8.5, color=INK2)
for ax in axs[1]: ax.set_xlabel("Horas")
for ax in axs[:, 0]: ax.set_ylabel("Registros (%)")
fig.suptitle("Figura 1. Histogramas de las variables de tiempo (truncados en 12 h)", y=1.0, fontsize=11)
fig.tight_layout(); plt.show()

**Lectura:** las esperas se concentran en cero y el cargue y el descargue tienen una cola larga hacia la derecha.
La barra final del cargue y del descargue corresponde a registros con **exactamente 12 h** (no a todo lo que supera 12 h),
un valor repetido que puede ser un tope de reporte.

## 6. Naturaleza de la carga y configuración del vehículo

Se comparan los tiempos por grupo. Como los tiempos son muy asimétricos se usa **Kruskal–Wallis** y, como con
cientos de miles de registros casi cualquier diferencia sale significativa, se mira el **tamaño del efecto**
`ε² = H / (n − 1)` (menor a 0,01: despreciable; 0,01 a 0,06: pequeño).

In [ ]:
# 9. Resumen por grupo
def resumen(g):
    return pd.Series({"Registros": len(g),
                      "Espera cargue (media h)": g.HORAS_ESPERA_CARGUE.mean(), "Espera cargue > 2 h (%)": g.ec2.mean() * 100,
                      "Espera descargue (media h)": g.HORAS_ESPERA_DESCARGUE.mean(), "Espera descargue > 2 h (%)": g.ed2.mean() * 100,
                      "Cargue (mediana h)": g.HORAS_CARGUE.median(), "Descargue (mediana h)": g.HORAS_DESCARGUE.median()})


nat = df.groupby("NATURALEZA").apply(resumen, include_groups=False)
nat = nat[nat["Registros"] >= 1000].sort_values("Registros", ascending=False)
cfg = df.groupby("CONFIGURACION").apply(resumen, include_groups=False).sort_values("Registros", ascending=False).head(6)
NAT_ORDER = list(nat.index)
print("Naturaleza de la carga (grupos con al menos 1 000 registros):")
display(nat.style.format({"Registros": "{:,.0f}"}, precision=2))
print("Configuración del vehículo (las 6 más frecuentes):")
display(cfg.style.format({"Registros": "{:,.0f}"}, precision=2))

In [ ]:
# 10. Diagramas de caja por naturaleza de la carga
fig, axs = plt.subplots(2, 2, figsize=(8, 6.2))
grp3 = ["Carga Normal", "Carga Peligrosa", "Refrigerada"]
for ax, c in zip(axs.ravel(), TIMES):
    data = [df.loc[df.NATURALEZA == g, c].values for g in grp3]
    bp = ax.boxplot(data, whis=(5, 95), showfliers=False, patch_artist=True, widths=0.55,
                    medianprops=dict(color=INK, lw=1.4), whiskerprops=dict(color=INK2, lw=0.9), capprops=dict(color=INK2, lw=0.9))
    for patch, col in zip(bp["boxes"], [BLUE, ORANGE, AQUA]):
        patch.set_facecolor(col); patch.set_edgecolor("white"); patch.set_alpha(0.9)
    ax.set_xticks([1, 2, 3]); ax.set_xticklabels(["Normal", "Peligrosa", "Refrigerada"])
    ax.set_title(LAB[c]); ax.grid(axis="x", visible=False)
    ax.set_ylim(0, max(np.percentile(q, 95) for q in data) * 1.12 + 0.1)
for ax in axs[:, 0]: ax.set_ylabel("Horas")
fig.suptitle("Figura 2. Diagramas de caja por naturaleza de la carga (bigotes en P5 y P95, sin atípicos)", y=1.0, fontsize=11)
fig.tight_layout(); plt.show()

In [ ]:
# 11. Pruebas de Kruskal-Wallis (efecto de la naturaleza y de la configuración)
filas = []
for var in TIMES:
    for col, grupos, nombre in [("NATURALEZA", NAT_ORDER, "Naturaleza de la carga"),
                                ("CONFIGURACION", list(cfg.index), "Configuración (6 principales)")]:
        t = kruskal(var, col, df, grupos)
        filas.append([nombre, LAB[var], t["H"], t["p"], t["eps2"]])
kw = pd.DataFrame(filas, columns=["Factor", "Variable", "H", "valor p", "ε²"])
kw["valor p"] = kw["valor p"].map(lambda p: "< 0.001" if p < 1e-3 else f"{p:.3f}")
display(kw.style.format({"H": "{:,.1f}", "ε²": "{:.3f}"}).hide(axis="index"))

**Lectura:** la carga peligrosa y la refrigerada tienen **operaciones más largas** (mediana de descargue cercana a 2 h
frente a 1 h en carga normal), pero no más esperas. Todas las diferencias son significativas (p muy pequeño) y de
efecto despreciable o pequeño (ε² ≤ 0,03), por lo que importan poco en la práctica.

## 7. Municipios de origen y destino

Se toman los 10 municipios con más registros como origen y como destino y se compara el porcentaje de esperas
de más de 2 h. Se usa chi-cuadrado con la **V de Cramér** como tamaño del efecto.

In [ ]:
# 12. Municipios
SUF = ["BOGOTA D. C.", "VALLE DEL CAUCA", "NORTE DE SANTANDER", "LA GUAJIRA", "ANTIOQUIA", "ATLANTICO", "BOLIVAR",
       "CUNDINAMARCA", "SANTANDER", "RISARALDA", "META", "CALDAS", "BOYACA", "MAGDALENA", "TOLIMA", "CORDOBA", "HUILA",
       "CESAR", "NARIÑO", "CAUCA", "QUINDIO", "CASANARE", "SUCRE", "CHOCO", "ARAUCA", "PUTUMAYO", "CAQUETA", "GUAVIARE",
       "VICHADA", "AMAZONAS"]
SUF.sort(key=len, reverse=True)


def corto(s):
    for x in SUF:
        if s.endswith(" " + x):
            s = s[: -len(x) - 1]; break
    s = s.title()
    return {"Medellin": "Medellín", "Bogota": "Bogotá", "Tocancipa": "Tocancipá"}.get(s, s)


top_o = df.CARGUE.value_counts().head(10).index
top_d = df.DESCARGUE.value_counts().head(10).index
go = df[df.CARGUE.isin(top_o)].groupby("CARGUE").agg(n=("ec2", "size"), p=("ec2", "mean"))
gd = df[df.DESCARGUE.isin(top_d)].groupby("DESCARGUE").agg(n=("ed2", "size"), p=("ed2", "mean"))
go["p"] *= 100; gd["p"] *= 100
go = go.sort_values("p"); gd = gd.sort_values("p")
print(f"Los 10 municipios principales concentran {df.CARGUE.isin(top_o).mean() * 100:.1f} % de los registros como origen "
      f"y {df.DESCARGUE.isin(top_d).mean() * 100:.1f} % como destino.")

for nombre, col, top, flag in [("origen", "CARGUE", top_o, "ec2"), ("destino", "DESCARGUE", top_d, "ed2")]:
    sub = df[df[col].isin(top)]
    ct = pd.crosstab(sub[col], sub[flag])
    chi2, p, dof, _ = stats.chi2_contingency(ct)
    V = np.sqrt(chi2 / (ct.values.sum() * (min(ct.shape) - 1)))
    print(f"Chi-cuadrado por {nombre}: χ²({dof}) = {chi2:,.0f}, p {pf(p)}, V de Cramér = {V:.3f}")

fig, axs = plt.subplots(2, 1, figsize=(7, 8.5))
for ax, g, tit, tot in [(axs[0], go, "Espera para cargue > 2 h, por origen", df.ec2.mean() * 100),
                        (axs[1], gd, "Espera para descargue > 2 h, por destino", df.ed2.mean() * 100)]:
    y = np.arange(len(g))
    ax.barh(y, g.p, color=BLUE, height=0.62)
    ax.set_yticks(y); ax.set_yticklabels([corto(i) for i in g.index])
    ax.axvline(tot, color=INK2, ls="--", lw=1)
    ax.set_title(f"{tit} (--- total {tot:.1f} %)", loc="left"); ax.grid(axis="y", visible=False)
    for yy, v in zip(y, g.p):
        ax.text(v + 0.15, yy, f"{v:.1f}", va="center", fontsize=8.5, color=INK2)
    ax.set_xlim(0, max(g.p.max(), tot) * 1.18)
axs[1].set_xlabel("Registros con espera > 2 h (%)")
fig.suptitle("Figura 3. Esperas largas en los diez municipios con más registros", y=1.0, fontsize=11)
fig.tight_layout(); plt.show()

for nombre, col, var, top in [("origen", "CARGUE", "HORAS_ESPERA_CARGUE", top_o), ("destino", "DESCARGUE", "HORAS_ESPERA_DESCARGUE", top_d)]:
    t = kruskal(var, col, df, list(top))
    print(f"Kruskal–Wallis por {nombre}: H = {t['H']:,.1f}, p {pf(t['p'])}, ε² = {t['eps2']:.3f}")

**Lectura:** como destino, Yumbo destaca con cerca del doble del promedio de esperas largas; como origen, Medellín
es el más alto y Bucaramanga el más bajo. La asociación existe pero es **débil** (V de Cramér ≈ 0,05–0,06).

## 8. Patrones por día y por hora

Primero el volumen y las esperas largas por día de enero (sombreado: domingos y festivos), luego el día de la semana y la
hora de llegada al sitio de descargue.

In [ ]:
# 13. Serie diaria (por fecha de salida del cargue)
dia = (df.assign(dia=df.fs.dt.day).dropna(subset=["dia"]).groupby("dia")
         .agg(n=("ec2", "size"), ec=("ec2", "mean"), ed=("ed2", "mean")))
dia.index = dia.index.astype(int)
dia["ec"] *= 100; dia["ed"] *= 100
domingos = [d for d in range(1, 32) if pd.Timestamp(2021, 1, d).dayofweek == 6]
festivos = [1, 11]
habiles = [d for d in range(1, 32) if d not in domingos + festivos]
n_dom = dia.loc[dia.index.isin(domingos), "n"]        # registros por día en domingos
n_hab = dia.loc[dia.index.isin(habiles), "n"]         # registros por día en días hábiles

fig, axs = plt.subplots(2, 1, figsize=(10, 6), sharex=True, gridspec_kw={"height_ratios": [1, 1.25]})
for ax in axs:
    for d in domingos: ax.axvspan(d - 0.5, d + 0.5, color=GRAY, alpha=0.28, lw=0)
    for d in festivos: ax.axvspan(d - 0.5, d + 0.5, color=ORANGE, alpha=0.18, lw=0)
axs[0].bar(dia.index, dia.n / 1000, color=BLUE, width=0.7); axs[0].set_ylabel("Registros (miles)")
axs[1].plot(dia.index, dia.ec, color=BLUE, lw=1.6, marker="o", ms=4)
axs[1].plot(dia.index, dia.ed, color=ORANGE, lw=1.6, marker="o", ms=4)
axs[1].set_ylabel("Espera > 2 h (%)"); axs[1].set_xlabel("Día de enero de 2021 (fecha de salida del cargue)")
axs[1].set_xlim(0.4, 31.6); axs[1].set_xticks(range(1, 32, 2)); axs[1].set_ylim(0)
axs[1].legend(handles=[plt.Line2D([], [], color=BLUE, lw=1.6, marker="o", ms=4, label="Para cargue"),
                       plt.Line2D([], [], color=ORANGE, lw=1.6, marker="o", ms=4, label="Para descargue"),
                       Patch(color=GRAY, alpha=0.4, label="Domingo"), Patch(color=ORANGE, alpha=0.25, label="Festivo")],
              ncol=4, loc="upper center", bbox_to_anchor=(0.5, 1.3), frameon=False)
fig.suptitle("Figura 4. Registros por día y porcentaje con espera > 2 h", y=1.02, fontsize=11)
fig.tight_layout(); plt.show()

print(f"Un domingo tiene en promedio {n_dom.mean():,.0f} registros, "
      f"{n_dom.mean() / n_hab.mean() * 100:.0f} % de un día hábil ({n_hab.mean():,.0f}).")
print(f"El día con menos registros es el {int(dia.n.idxmin())} de enero ({int(dia.n.min()):,}).")

dom = df.dow_s == 6
u = stats.mannwhitneyu(df.loc[dom, "HORAS_ESPERA_CARGUE"], df.loc[~dom, "HORAS_ESPERA_CARGUE"], alternative="two-sided", method="asymptotic")
print(f"Salida en domingo vs resto: espera > 2 h para cargar {df.loc[dom, 'ec2'].mean() * 100:.1f} % vs {df.loc[~dom, 'ec2'].mean() * 100:.1f} %; "
      f"Mann–Whitney p {pf(u.pvalue)}, prob. de superioridad = {u.statistic / (dom.sum() * (~dom).sum()):.3f} (0,5 = sin diferencia en el centro).")

In [ ]:
# 14. Día de la semana y hora de llegada al descargue
wc = df.groupby("dow_s").ec2.mean() * 100
wd = df.groupby("dow_l").ed2.mean() * 100
hh = df.groupby("h_ll").agg(n=("ed2", "size"), p=("ed2", "mean"))
hh["p"] *= 100

fig, axs = plt.subplots(1, 2, figsize=(12, 3.8), gridspec_kw={"width_ratios": [1, 1.55]})
ax = axs[0]
x = np.arange(7); w = 0.38
ax.bar(x - w / 2, wc.values, w, color=BLUE, label="Para cargue (día de salida)")
ax.bar(x + w / 2, wd.values, w, color=ORANGE, label="Para descargue (día de llegada)")
ax.set_xticks(x); ax.set_xticklabels(DIAS); ax.grid(axis="x", visible=False)
ax.set_ylabel("Registros con espera > 2 h (%)"); ax.set_title("(a) Día de la semana", loc="left")
ax.legend(frameon=False, loc="upper left"); ax.set_ylim(0, max(wc.max(), wd.max()) * 1.38)
ax = axs[1]
tot = df.ed2.mean() * 100
ax.bar(hh.index, hh.p, color=BLUE, width=0.72)
ax.axhline(tot, color=INK2, ls="--", lw=1)
ax.text(23.4, tot + 0.25, f"total {tot:.1f} %", ha="right", fontsize=8.5, color=INK2)
ax.set_xticks(range(0, 24, 2)); ax.set_xlim(-0.7, 23.7)
ax.set_xlabel("Hora de llegada al sitio de descargue"); ax.grid(axis="x", visible=False)
ax.set_ylabel("Espera para descargue > 2 h (%)"); ax.set_title("(b) Hora de llegada", loc="left")
fig.suptitle("Figura 5. Esperas largas por día de la semana y por hora de llegada", y=1.03, fontsize=11)
fig.tight_layout(); plt.show()

noche = df.h_ll.between(3, 7)
print(f"Llegadas entre 3:00 y 7:59 h: {noche.mean() * 100:.1f} % de los registros.")
print(f"  Espera > 2 h para descargar: {df.loc[noche, 'ed2'].mean() * 100:.1f} % frente a {df.loc[~noche, 'ed2'].mean() * 100:.1f} % en el resto del día.")
print(f"  Espera media: {df.loc[noche, 'HORAS_ESPERA_DESCARGUE'].mean():.2f} h frente a {df.loc[~noche, 'HORAS_ESPERA_DESCARGUE'].mean():.2f} h.")
print(f"  Máximo a las {int(hh.p.idxmax())} h ({hh.p.max():.1f} %), mínimo a las {int(hh.p.idxmin())} h ({hh.p.min():.1f} %): razón {hh.p.max() / hh.p.min():.1f} veces.")
t = kruskal("HORAS_ESPERA_DESCARGUE", "h_ll", df)
print(f"  Kruskal–Wallis por hora: H = {t['H']:,.0f}, p {pf(t['p'])}, ε² = {t['eps2']:.3f} (el mayor efecto entre los factores, aunque pequeño).")

**Lectura:** los domingos y festivos hay muchos menos camiones y más esperas largas, pero el cambio ocurre en la cola
y no en el centro de la distribución. La **hora de llegada** es el patrón más marcado: entre las 3:00 y las 7:59 h la
proporción de esperas largas se duplica. Es coherente con vehículos que llegan antes de que abra el sitio, pero la base
no trae el horario de cada sitio, así que es solo una hipótesis.

## 9. Relaciones entre variables

Se usa la correlación de **Spearman** (basada en rangos) porque las variables son muy asimétricas, tienen muchos ceros
y valores repetidos, y no se espera una relación lineal. La cantidad solo incluye los viajes medidos en kilogramos.

In [ ]:
# 15. Matriz de correlación de Spearman
cm = df[["HORAS_ESPERA_CARGUE", "HORAS_CARGUE", "HORAS_ESPERA_DESCARGUE", "HORAS_DESCARGUE", "HORAS_VIAJE",
         "CANTIDAD_REMESAS_VIAJE"]].copy()
cm["CANTIDAD_KG"] = np.where(df.UNID_MEDIDA == "Kilogramos", df.CANTIDAD, np.nan)
corr = cm.corr(method="spearman")
nombres = ["Esp. carg.", "Cargue", "Esp. desc.", "Descargue", "Viaje", "Remesas", "Cant. (kg)"]
C = corr.values

cmap = LinearSegmentedColormap.from_list("div", [BLUE, "#f2f1ee", ORANGE])
fig, ax = plt.subplots(figsize=(6.5, 5.8))
im = ax.imshow(C, cmap=cmap, vmin=-1, vmax=1)
ax.set_xticks(range(7)); ax.set_yticks(range(7))
ax.set_xticklabels(nombres, rotation=45, ha="right"); ax.set_yticklabels(nombres); ax.grid(False)
for s in ax.spines.values(): s.set_visible(False)
for i in range(7):
    for j in range(7):
        ax.text(j, i, f"{C[i, j]:.2f}", ha="center", va="center", fontsize=9, color=INK if abs(C[i, j]) < 0.7 else "white")
cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03); cb.outline.set_visible(False)
ax.set_title("Figura 6. Correlación de Spearman")
fig.tight_layout(); plt.show()

dv0 = df.drop_duplicates(subset=["PLACA", "FECHASALIDACARGUE", "CODIGO_CARGUE", "CODIGO_DESCARGUE"])
rv = dv0[["HORAS_ESPERA_CARGUE", "HORAS_ESPERA_DESCARGUE"]].corr(method="spearman").iloc[0, 1]
p1 = df.loc[df.ec2, "ed2"].mean() * 100
p0 = df.loc[~df.ec2, "ed2"].mean() * 100
print(f"Espera para cargue vs espera para descargue: ρ = {C[0, 2]:.2f} (con un registro por viaje: {rv:.2f}).")
print(f"P(espera > 2 h al descargar | espera > 2 h al cargar) = {p1:.1f} %, frente a {p0:.1f} % si no hubo: {p1 / p0:.1f} veces más.")

**Lectura:** la espera para cargue y la de descargue se asocian con fuerza (ρ ≈ 0,75), y esto no se debe a remesas
repetidas del mismo viaje. En cambio, las esperas casi no se relacionan con los tiempos de operación, las horas de viaje, las
remesas ni los kilogramos (|ρ| ≤ 0,12): **no se explican por cuánto se transporta**. La asociación entre esperas
sugiere un factor común al viaje (la empresa, el tipo de operación o la forma de reportar); no significa que una
espera cause la otra.

## 10. Concentración por empresa y valor pagado

In [ ]:
# 16. Concentración de esperas largas por empresa
for nombre, flag in [("descargue", "ed2"), ("cargue", "ec2")]:
    e = df.groupby("EMPRESA_TRANSPORTE").agg(n=(flag, "size"), k=(flag, "sum")).sort_values("k", ascending=False)
    top = e.head(20)
    gran = e[e.n >= 100]
    print(f"Espera para {nombre}: las 20 empresas con más esperas largas reúnen {top.k.sum() / e.k.sum() * 100:.1f} % de ellas "
          f"con solo {top.n.sum() / e.n.sum() * 100:.1f} % de los registros.")
    print(f"   Entre las {len(gran)} empresas con al menos 100 registros, {int(((gran.k / gran.n) >= 0.9).sum())} tienen "
          f"espera > 2 h en al menos el 90 % de sus registros.")

In [ ]:
# 17. Valor pactado vs valor pagado (solo viajes de una remesa con valor pactado >= 100 000)
print(f"Valor pactado igual a 0: {(df.VALOR_PACTADO == 0).mean() * 100:.1f} % de los registros; "
      f"menor a 1 000: {(df.VALOR_PACTADO < 1000).mean() * 100:.1f} %  (no es confiable por remesa).")
v = df[(df.CANTIDAD_REMESAS_VIAJE == 1) & (df.VALOR_PACTADO >= 100000)].copy()
v["rel"] = v.VALOR_PAGADO / v.VALOR_PACTADO - 1
v["igual"] = v.rel.abs() <= 0.001
v["mas"] = v.rel > 0.001
v["Espera total (h)"] = pd.cut(v.espera_total, [-0.1, 0, 2, 6, 12, 1e9], labels=["0", "0-2", "2-6", "6-12", "> 12"])
vt = v.groupby("Espera total (h)", observed=True).agg(Viajes=("igual", "size"), **{"% pagado = pactado": ("igual", "mean"),
                                                                                        "% pagado > pactado": ("mas", "mean")})
vt.iloc[:, 1:] *= 100
print(f"Viajes comparables: {len(v):,}. Pagado exactamente igual al pactado: {v.igual.mean() * 100:.1f} %; mayor: {v['mas'].mean() * 100:.1f} %.")
display(vt.style.format({"Viajes": "{:,.0f}", "% pagado = pactado": "{:.1f}", "% pagado > pactado": "{:.1f}"}))

**Lectura:** las esperas largas están concentradas en pocas empresas, lo que puede reflejar operaciones realmente distintas
o prácticas de reporte (con estos datos no se pueden separar). Casi todos los viajes se pagan exactamente lo pactado,
sin relación con la espera: la base no registra pagos por demora aparte.

## 11. Sensibilidad de los resultados

¿Cambian las conclusiones si se limpia de otra forma? Se comparan la base limpia con: conservar los duplicados, usar un
registro por viaje, quitar valores por encima del percentil 99 y quitar los atípicos por IQR.

In [ ]:
# 18. Variantes de limpieza
def resumen_s(d):
    return {"Registros": len(d),
            "Espera cargue media (h)": d.HORAS_ESPERA_CARGUE.mean(), "Espera cargue > 2 h (%)": (d.HORAS_ESPERA_CARGUE > 2).mean() * 100,
            "Espera descargue media (h)": d.HORAS_ESPERA_DESCARGUE.mean(), "Espera descargue > 2 h (%)": (d.HORAS_ESPERA_DESCARGUE > 2).mean() * 100,
            "Cargue mediana (h)": d.HORAS_CARGUE.median(), "Descargue mediana (h)": d.HORAS_DESCARGUE.median()}


p99 = {c: df[c].quantile(.99) for c in TIMES}
variantes = {
    "Base (limpia)": df,
    "Con duplicados": dfB,
    "Un registro por viaje": df.drop_duplicates(subset=["PLACA", "FECHASALIDACARGUE", "CODIGO_CARGUE", "CODIGO_DESCARGUE"]),
    "Sin valores > P99": df[np.logical_and.reduce([df[c] <= p99[c] for c in TIMES])],
    "Sin atípicos IQR": df[~out_any],
}
sens = pd.DataFrame({k: resumen_s(d) for k, d in variantes.items()}).T
display(sens.style.format({"Registros": "{:,.0f}"}, precision=2))
print("Con IQR no queda ninguna espera > 2 h por construcción: esa variante no sirve para estudiar esperas largas.")

dI = df[~out_any]
for var in TIMES:
    t = kruskal(var, "NATURALEZA", dI, NAT_ORDER)
    print(f"Naturaleza de la carga sin atípicos IQR, {LAB[var]}: ε² = {t['eps2']:.3f}")

## 12. Conclusiones

- **En el registro típico no hay espera, pero la cola es larga:** alrededor del 4 % de los registros espera más de 2 h para cargar
  y cerca del 5 % para descargar; el cargue y el descargue superan 2 h en más del 20 % de los casos.
- **Qué se asocia con las esperas largas:** la hora de llegada (entre las 3 y las 7 h), los domingos y festivos, destinos como Yumbo
  y pocas empresas. Los efectos por tipo de carga, configuración y municipio son significativos pero pequeños (ε² ≤ 0,03).
- **Qué no se asocia:** el volumen (kilogramos, remesas) ni el tiempo de viaje. La espera para cargue y la de descargue sí se
  asocian (ρ ≈ 0,75), lo que apunta a factores comunes al viaje.
- **Recomendaciones:** evaluar la programación de llegadas con ventanas de tiempo en los sitios con más esperas y verificar con
  las empresas con más esperas si su reporte refleja la operación.
- **Limitaciones:** un solo mes (enero de 2021), tiempos reportados y redondeados, sin horario del sitio ni tiempo pactado; son
  asociaciones, no causas.